# Random Forest Results Analysis

This notebook mirrors the style of the `rnn-lstm-results.py` analysis workflow, but for the MPI + OpenMP Random Forest project.

It focuses on:
- comparing the latest Serial, OpenMP, MPI, and Hybrid runs
- visualizing validation metrics such as accuracy, precision, recall, F1, and overhead
- comparing runtime across implementations
- summarizing multi-seed hybrid experiments

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
if (BASE_DIR / "runs").exists():
    RESULTS_DIR = BASE_DIR
elif (BASE_DIR / "Results" / "runs").exists():
    RESULTS_DIR = BASE_DIR / "Results"
else:
    raise FileNotFoundError("Could not find Results/runs from the current working directory.")

RUNS_DIR = RESULTS_DIR / "runs"
RUNS_DIR

In [ ]:
def load_runs(runs_dir: Path) -> pd.DataFrame:
    rows = []
    for metrics_path in sorted(runs_dir.glob("run_*/metrics.json")):
        with metrics_path.open("r", encoding="utf-8") as infile:
            data = json.load(infile)
        data["run_name"] = metrics_path.parent.name
        data["metrics_path"] = str(metrics_path)
        rows.append(data)
    if not rows:
        raise ValueError("No run metrics found under Results/runs/run_*/metrics.json")
    return pd.DataFrame(rows)

runs_df = load_runs(RUNS_DIR)
runs_df.head()

## Available Runs

The table below shows the saved experiment artifacts discovered under `Results/runs/`.

In [ ]:
display_columns = [
    "model_name", "split_seed", "n_trees", "max_depth",
    "min_samples_split", "max_features", "mpi_ranks",
    "omp_threads", "accuracy", "precision", "recall",
    "f1", "runtime_sec", "run_name"
]
runs_df[display_columns].sort_values(["model_name", "run_name"]).tail(20)

## Latest Baseline Runs

This section compares the latest run artifact for each of the main baseline model families:
- Backup Serial
- Backup OpenMP
- Backup MPI
- Hybrid MPI + OpenMP

In [ ]:
baseline_models = [
    "Random Forest (Backup Serial)",
    "Random Forest (Backup OpenMP)",
    "Random Forest (Backup MPI)",
    "Random Forest (Hybrid)",
]

baseline_df = (
    runs_df[runs_df["model_name"].isin(baseline_models)]
    .sort_values("run_name")
    .groupby("model_name", as_index=False)
    .tail(1)
    .copy()
)

baseline_df = baseline_df.set_index("model_name").loc[
    [m for m in baseline_models if m in set(baseline_df["model_name"])]
].reset_index()

baseline_df[[
    "model_name", "accuracy", "precision", "recall", "f1",
    "overhead", "runtime_sec", "mpi_ranks", "omp_threads", "run_name"
]]

In [ ]:
metric_columns = ["accuracy", "precision", "recall", "f1", "overhead"]
fig, axes = plt.subplots(1, len(metric_columns), figsize=(22, 5), constrained_layout=True)

for ax, metric in zip(axes, metric_columns):
    ax.bar(baseline_df["model_name"], baseline_df[metric], color=["#4C78A8", "#72B7B2", "#F58518", "#54A24B"])
    ax.set_title(metric.replace("_", " ").title())
    ax.set_ylabel("Percent")
    ax.tick_params(axis="x", rotation=35)
    for idx, value in enumerate(baseline_df[metric]):
        ax.text(idx, value, f"{value:.2f}", ha="center", va="bottom", fontsize=9)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(baseline_df["model_name"], baseline_df["runtime_sec"], color=["#4C78A8", "#72B7B2", "#F58518", "#54A24B"])
plt.title("Runtime Comparison Across Latest Baseline Runs")
plt.ylabel("Runtime (seconds)")
plt.xticks(rotation=35, ha="right")
for idx, value in enumerate(baseline_df["runtime_sec"]):
    plt.text(idx, value, f"{value:.4f}", ha="center", va="bottom")
plt.show()

## Compare & Contrast: Serial vs OpenMP vs MPI vs Hybrid

- **Backup Serial:** This version provides the cleanest single-process baseline for the older Random Forest implementation. It is useful for understanding the original model behavior, but its validation F1 is much lower than the current hybrid implementation under the same held-out workflow.
- **Backup OpenMP:** This version parallelizes tree training on one process using threads. It tends to run faster than the backup serial implementation, but it does not automatically produce better validation quality.
- **Backup MPI:** This version shows what the older Random Forest pipeline looks like under MPI-style distributed tree ownership. It is helpful as a process-level baseline, but its validation performance still trails the current hybrid implementation.
- **Hybrid MPI + OpenMP:** This is the strongest current implementation in the project. The gain is not from parallelism alone; it comes from the newer workflow, histogram/binning design, corrected vote aggregation, and cleaner train-only preprocessing.

## Hybrid Multi-Seed Analysis

This section focuses on saved Hybrid runs only and helps compare repeated experiments across split seeds.

In [ ]:
hybrid_df = runs_df[runs_df["model_name"] == "Random Forest (Hybrid)"].copy()
hybrid_df[[
    "split_seed", "n_trees", "max_depth", "min_samples_split",
    "max_features", "accuracy", "precision", "recall", "f1", "runtime_sec", "run_name"
]].sort_values(["n_trees", "max_depth", "min_samples_split", "max_features", "split_seed"]).tail(20)

In [ ]:
group_columns = [
    "n_trees", "max_depth", "min_samples_split", "max_features",
    "train_fraction", "mpi_ranks"
]

hybrid_summary = (
    hybrid_df.groupby(group_columns)
    .agg(
        run_count=("f1", "count"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        runtime_mean=("runtime_sec", "mean"),
    )
    .reset_index()
    .sort_values("f1_mean", ascending=False)
)

hybrid_summary.head(10)

In [ ]:
best_config = hybrid_summary.iloc[0]
best_runs = hybrid_df[
    (hybrid_df["n_trees"] == best_config["n_trees"]) &
    (hybrid_df["max_depth"] == best_config["max_depth"]) &
    (hybrid_df["min_samples_split"] == best_config["min_samples_split"]) &
    (hybrid_df["max_features"] == best_config["max_features"]) &
    (hybrid_df["train_fraction"] == best_config["train_fraction"]) &
    (hybrid_df["mpi_ranks"] == best_config["mpi_ranks"])
].sort_values("split_seed")

best_runs[["split_seed", "accuracy", "precision", "recall", "f1", "runtime_sec", "run_name"]]

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(best_runs["split_seed"], best_runs["f1"], marker="o", label="F1")
plt.plot(best_runs["split_seed"], best_runs["accuracy"], marker="s", label="Accuracy")
plt.title("Hybrid Multi-Seed Stability for Best Current Configuration")
plt.xlabel("Split Seed")
plt.ylabel("Validation Metric (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Compare & Contrast: Hybrid Multi-Seed Results

- The goal here is not only to find the highest single F1 score, but also to understand how stable the hybrid configuration is across different split seeds.
- A strong configuration should have both a competitive mean F1 and a manageable standard deviation.
- If two configurations perform similarly, the lower-variance and lower-runtime configuration is usually the better default choice.

## Suggested Workflow

1. Run fresh experiments from the terminal.
2. Save per-run `metrics.json` artifacts under `Results/runs/`.
3. Open this notebook and rerun all cells.
4. Compare the latest baseline runs and then inspect hybrid multi-seed stability.

Useful commands:

```bash
make compare-backup-baselines
make sweep-seeds NP=2 SEEDS="7 21 42 84 123" RUN_ARGS="--trees 300 --max-depth 12 --min-samples-split 10"
make summarize-runs
make visualize
```

## Conclusions

- **Best current Hybrid direction:** The strongest Hybrid configuration observed so far uses a validation-based workflow with stratified splitting, train-only preprocessing, and tuned Random Forest hyperparameters rather than relying on the older in-sample evaluation path.
- **Parallelism vs model quality:** MPI and OpenMP do not automatically improve model quality by themselves. The major gains came from the corrected evaluation workflow, train-only imputation / preprocessing, and better Random Forest configuration choices.
- **Backup baseline comparison:** The repaired Backup Serial, OpenMP, and MPI baselines are useful for comparison, but the current Hybrid implementation is the strongest overall reference because it combines the cleaner experiment workflow with the newer Random Forest implementation.
- **How to interpret results:** Use runtime plots to study performance differences, and use validation accuracy / recall / F1 plots to study predictive behavior. F1 and recall are especially important here because the dataset is not perfectly balanced.
- **Recommended final reporting pattern:** For conclusions in reports or presentations, emphasize multi-seed averages, not just a single split seed. If two configurations are close, prefer the one with lower variance and lower runtime.